# Model Training pipeline 

In [34]:
import pandas as pd
import numpy as np
from datetime import datetime 
import pickle 
import os
import warnings
warnings.filterwarnings('ignore')

# ML librararies
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb

# Metrics
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, 
    f1_score, confusion_matrix, classification_report, log_loss
)

# Helper Function (Brier score)

In [35]:
def compute_brier_score(y_true, y_prob):
    """
    Compute the Brier score for probabilistic predictions.
    
    Parameters:
    y_true (array-like): True binary labels (0 or 1).
    y_prob (array-like): Predicted probabilities for the positive class.
    
    Returns:
    float: Brier score.
    """
    classes=np.unique(y_true)
    n_classes=len(classes)
    class_to_idx={cls:idx for idx, cls in enumerate(classes)}

    # one hot encode true labels
    y_true_onehot=np.zeros((len(y_true), n_classes))
    for i, label in enumerate(y_true):
        y_true_onehot[i, class_to_idx[label]] = 1
    
    brier=np.mean(np.sum((y_prob- y_true_onehot)**2, axis=1))
    return brier

# Load and prepare the data

In [36]:
# Load and prepare the data 
df=pd.read_csv(os.path.join('..', 'Processed', 'engineered_data.csv'))
print(f"Dataset shape: {df.shape}")
df['date']=pd.to_datetime(df['date'])

print(f"Target variable distribution: ")
print(df['outcome'].value_counts().sort_index())
print("\nClass imbalance: \n")
for outcome, label in [(-1, 'Loss'), (0, 'Draw'), (1, 'Win')]:
    pct=(df['outcome']==outcome).sum()/len(df)*100
    print(f"{label}: {pct:.2f}%")

Dataset shape: (4644, 36)
Target variable distribution: 
outcome
-1    1688
 0    1261
 1    1695
Name: count, dtype: int64

Class imbalance: 

Loss: 36.35%
Draw: 27.15%
Win: 36.50%


# Feature Selection and Train/Test Split (Time based)

In [37]:
# Features to exclude 
exclude_cols = [
    'date', 'outcome', 'team', 'opponent', 'day_of_week', 'comp', 'round', 'time'
]
feature_cols = [col for col in df.columns if col not in exclude_cols]
print(f"\nFeature selected for modeling: {feature_cols}")

rolling_features = [col for col in feature_cols if 'avg_' in col]
derived_features = [col for col in feature_cols if any(x in col for x in [
    'avg_goal_diff', 'avg_xgoal_diff', 'avg_shot_acc', 'avg_pk_acc', 'form_', 'points_'
])]
categorical_features = [col for col in feature_cols if '_encoded' in col or col == 'is_home']
temporal_features = [col for col in feature_cols if col in ['year', 'month', 'season', 'day_of_week_encoded']]
other_features = [col for col in feature_cols if col not in rolling_features + derived_features + categorical_features + temporal_features]

print(f"  Rolling averages: {len(rolling_features)}")
print(f"  Derived features: {len(derived_features)}")
print(f"  Categorical (encoded): {len(categorical_features)}")
print(f"  Temporal: {len(temporal_features)}")
print(f"  Other: {len(other_features)}")

# Sort by date to maintain temporal order
df_sorted = df.sort_values('date').reset_index(drop=True)

# Prepare X and y
X = df_sorted[feature_cols].copy()
y = df_sorted['outcome'].copy()

print(f"\nFeature matrix shape: {X.shape}")
print(f"Target vector shape: {y.shape}")

# Time based train-test split
split_idx = int(len(df_sorted) * 0.8)

train_dates = df_sorted['date'].iloc[:split_idx]
test_dates  = df_sorted['date'].iloc[split_idx:]

print(f"\nTime based train-test split (80/20):\n")
print(f"Training sets : {train_dates.min()} to {train_dates.max()}")
print(f"Testing sets: {test_dates.min()} to {test_dates.max()}")

# Get train/test indices  ✅ use df_sorted here, not df
train_mask = df_sorted['date'] <= train_dates.max()
test_mask  = df_sorted['date'] > train_dates.max()

X_train, X_test = X[train_mask].copy(), X[test_mask].copy()
y_train, y_test = y[train_mask].copy(), y[test_mask].copy()

print(f"Training set shape: {X_train.shape[0]} samples ({X_train.shape[0] / len(df_sorted) * 100:.2f}%)")
print(f"Testing set shape: {X_test.shape[0]} samples ({X_test.shape[0] / len(df_sorted) * 100:.2f}%)")
print("\n")
print(f"Class distribution in training set: {y_train.value_counts().sort_index()}")
print(f"Class distribution in testing set: {y_test.value_counts().sort_index()}")


Feature selected for modeling: ['season', 'year', 'month', 'team_encoded', 'opponent_encoded', 'venue_encoded', 'formation_encoded', 'opp formation_encoded', 'is_home', 'avg_gf_5', 'avg_ga_5', 'avg_xg_5', 'avg_xga_5', 'avg_poss_5', 'avg_sh_5', 'avg_sot_5', 'avg_dist_5', 'avg_fk_5', 'avg_pk_5', 'avg_pkatt_5', 'avg_goal_diff', 'avg_xgoal_diff', 'avg_shot_acc', 'avg_pk_acc', 'form_5', 'points_5', 'attendance', 'day_of_week_encoded']
  Rolling averages: 15
  Derived features: 6
  Categorical (encoded): 7
  Temporal: 4
  Other: 1

Feature matrix shape: (4644, 28)
Target vector shape: (4644,)

Time based train-test split (80/20):

Training sets : 2019-08-30 00:00:00 to 2024-05-19 00:00:00
Testing sets: 2024-05-19 00:00:00 to 2025-09-30 00:00:00
Training set shape: 3726 samples (80.23%)
Testing set shape: 918 samples (19.77%)


Class distribution in training set: outcome
-1    1351
 0    1019
 1    1356
Name: count, dtype: int64
Class distribution in testing set: outcome
-1    337
 0    242


# Feature Scalling for the models that need it 

In [38]:
# Feature Scaling
scaler=StandardScaler()
X_train_scaled=scaler.fit_transform(X_train)
X_test_scaled=scaler.transform(X_test)

print(f"Scaled trained shape: {X_train_scaled.shape}")
print(f"Scaled test shape: {X_test_scaled.shape}")

Scaled trained shape: (3726, 28)
Scaled test shape: (918, 28)


# Model Training and Evaluation

In [39]:
# Store results
results=[]
models={}
confusion_matrices={}

# Model 1: Logistic Regression

In [40]:
# Logistic Regression

lr_model=LogisticRegression(
    multi_class='multinomial',
    random_state=42,
    max_iter=1000,
    solver='lbfgs',
    class_weight='balanced'
)
lr_model.fit(X_train_scaled, y_train)
# Prediction
y_pred_lr=lr_model.predict(X_test_scaled)
y_pred_prob_lr=lr_model.predict_proba(X_test_scaled)
# Metrics
accuracy_lr=accuracy_score(y_test, y_pred_lr)
precision_lr=precision_score(y_test, y_pred_lr, average='weighted')
f1_macro_lr=f1_score(y_test, y_pred_lr, average='macro')
f1_weighted_lr=f1_score(y_test, y_pred_lr, average='weighted')
recall_lr=recall_score(y_test, y_pred_lr, average='weighted')
logg_loss_lr=log_loss(y_test, y_pred_prob_lr)
brier_score_lr=compute_brier_score(y_test.values, y_pred_prob_lr)
print("\nLogistic Regression Performance:")
print(f"Accuracy: {accuracy_lr:.4f}")
print(f"Precision: {precision_lr:.4f}")
print(f"F1 Score (Macro): {f1_macro_lr:.4f}")
print(f"F1 Score (Weighted): {f1_weighted_lr:.4f}")
print(f"Recall: {recall_lr:.4f}")
print(f"Log Loss: {logg_loss_lr:.4f}")
print(f"Brier Score: {brier_score_lr:.4f}")

class_report_lr=classification_report(y_test, y_pred_lr, target_names=['Loss', 'Draw', 'Win'])
print("\nClassification Report:\n")
print(class_report_lr)
# Confusion Matrix
cm_lr=confusion_matrix(y_test, y_pred_lr, labels=[-1,0,1])
confusion_matrices['Logistic Regression']=cm_lr
results.append({
    'Model': 'Logistic Regression',
    'Accuracy': accuracy_lr,
    'Precision': precision_lr,
    'F1 Score (Macro)': f1_macro_lr,
    'F1 Score (Weighted)': f1_weighted_lr,
    'Recall': recall_lr,
    'Log Loss': logg_loss_lr,
    'Brier Score': brier_score_lr
})
models['Logistic Regression']=lr_model


Logistic Regression Performance:
Accuracy: 0.4499
Precision: 0.4373
F1 Score (Macro): 0.4231
F1 Score (Weighted): 0.4422
Recall: 0.4499
Log Loss: 1.0467
Brier Score: 0.6292

Classification Report:

              precision    recall  f1-score   support

        Loss       0.48      0.52      0.50       337
        Draw       0.27      0.21      0.24       242
         Win       0.51      0.55      0.53       339

    accuracy                           0.45       918
   macro avg       0.42      0.43      0.42       918
weighted avg       0.44      0.45      0.44       918



# Model 2: Random Forest

In [41]:
# Random Forest Classifier
rf_model=RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_split=20,
    min_samples_leaf=10,
    random_state=42,
    n_jobs=-1, 
    class_weight='balanced'
)
rf_model.fit(X_train, y_train)

# Prediction
y_pred_rf=rf_model.predict(X_test)
y_pred_proba_rf=rf_model.predict_proba(X_test)

# Metrics

accruracy_rf=accuracy_score(y_test, y_pred_rf)
precision_rf=precision_score(y_test, y_pred_rf, average='weighted')
recall_rf=recall_score(y_test, y_pred_rf, average='weighted')
f1_macro_rf=f1_score(y_test, y_pred_rf, average='macro')
f1_weighted_rf=f1_score(y_test, y_pred_rf, average='weighted')
logg_loss_rf=log_loss(y_test, y_pred_proba_rf)
brier_score_rf=compute_brier_score(y_test.values, y_pred_proba_rf)
print("\nRandom Forest Classifier Performance:")
print(f"Accuracy: {accruracy_rf:.4f}")
print(f"Precision: {precision_rf:.4f}")
print(f"F1 Score (Macro): {f1_macro_rf:.4f}")
print(f"F1 Score (Weighted): {f1_weighted_rf:.4f}")
print(f"Recall: {recall_rf:.4f}")
print(f"Log Loss: {logg_loss_rf:.4f}")
print(f"Brier Score: {brier_score_rf:.4f}")

class_report_rf=classification_report(y_test, y_pred_rf, target_names=['Loss', 'Draw', 'Win'])
print("\nClassification Report:\n")
print(class_report_rf)

# Confusion Matrix
cm_rf=confusion_matrix(y_test, y_pred_rf, labels=[-1,0,1])
confusion_matrices['Random Forest']=cm_rf
# Feature Importance
print("\nTop 10 Most Important Features:")
feature_importance_rf = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)
print(feature_importance_rf.head(10).to_string(index=False))

results.append({
    'Model': 'Random Forest',
    'Accuracy': accruracy_rf,
    'Precision': precision_rf,
    'F1 Score (Macro)': f1_macro_rf,
    'F1 Score (Weighted)': f1_weighted_rf,
    'Recall': recall_rf,
    'Log Loss': logg_loss_rf,
    'Brier Score': brier_score_rf
})
models['Random Forest']=rf_model


Random Forest Classifier Performance:
Accuracy: 0.4662
Precision: 0.4388
F1 Score (Macro): 0.4227
F1 Score (Weighted): 0.4476
Recall: 0.4662
Log Loss: 1.0204
Brier Score: 0.6119

Classification Report:

              precision    recall  f1-score   support

        Loss       0.50      0.55      0.53       337
        Draw       0.24      0.15      0.19       242
         Win       0.52      0.60      0.56       339

    accuracy                           0.47       918
   macro avg       0.42      0.44      0.42       918
weighted avg       0.44      0.47      0.45       918


Top 10 Most Important Features:
         feature  importance
      avg_poss_5    0.079764
      attendance    0.071640
opponent_encoded    0.065323
  avg_xgoal_diff    0.060090
        avg_xg_5    0.054351
      avg_dist_5    0.054075
    avg_shot_acc    0.053992
        avg_sh_5    0.049424
       avg_xga_5    0.048856
       avg_sot_5    0.040169


# Model 3: XGBoost 

In [42]:
y_train_xgb=y_train.map({-1:0, 0:1, 1:2})
y_test_xgb=y_test.map({-1:0, 0:1, 1:2})

# XGBoost Classifier
xgb_model=xgb.XGBClassifier(
    n_estimators=200, 
    max_depth=6,
    learning_rate=0.1,
    objective='multi:softprob',
    num_class=3,
    random_state=42,
    subsample=0.8,
    colsample_bytree=0.8,
    n_jobs=-1,
)
xgb_model.fit(X_train, y_train_xgb)

# Prediction
y_pred_xgb=xgb_model.predict(X_test)
y_pred_proba_xgb=xgb_model.predict_proba(X_test)

# Map predictions back to original labels
y_pred_xgb_mapped=pd.Series(y_pred_xgb).map({0:-1, 1:0, 2:1})

# Metrics
accuracy_xgb=accuracy_score(y_test, y_pred_xgb_mapped)
precision_xgb=precision_score(y_test, y_pred_xgb_mapped, average='weighted')
recall_xgb=recall_score(y_test, y_pred_xgb_mapped, average='weighted')
f1_macro_xgb=f1_score(y_test, y_pred_xgb_mapped, average='macro')
f1_weighted_xgb=f1_score(y_test, y_pred_xgb_mapped, average='weighted')
logg_loss_xgb=log_loss(y_test_xgb, y_pred_proba_xgb)
brier_score_xgb=compute_brier_score(y_test.values, y_pred_proba_xgb)
print("\nXGBoost Classifier Performance:")
print(f"Accuracy: {accuracy_xgb:.4f}")
print(f"Precision: {precision_xgb:.4f}")
print(f"F1 Score (Macro): {f1_macro_xgb:.4f}")
print(f"F1 Score (Weighted): {f1_weighted_xgb:.4f}")
print(f"Recall: {recall_xgb:.4f}")
print(f"Log Loss: {logg_loss_xgb:.4f}")
print(f"Brier Score: {brier_score_xgb:.4f}")
class_report_xgb=classification_report(y_test, y_pred_xgb_mapped, target_names=['Loss', 'Draw', 'Win'])
print("\nClassification Report:\n")
print(class_report_xgb)

# Confusion Matrix
cm_xgb=confusion_matrix(y_test, y_pred_xgb_mapped, labels=[-1,0,1])
confusion_matrices['XGBoost']=cm_xgb

print("\nTop 10 Most Important Features:")
feature_importance_xgb = pd.DataFrame({
    'feature': feature_cols,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False)
print(feature_importance_xgb.head(10).to_string(index=False))

results.append({
    'Model': 'XGBoost',
    'Accuracy': accuracy_xgb,
    'Precision': precision_xgb,
    'F1 Score (Macro)': f1_macro_xgb,
    'F1 Score (Weighted)': f1_weighted_xgb,
    'Recall': recall_xgb,
    'Log Loss': logg_loss_xgb,
    'Brier Score': brier_score_xgb
})  
models['XGBoost']=xgb_model


XGBoost Classifier Performance:
Accuracy: 0.4858
Precision: 0.4595
F1 Score (Macro): 0.4403
F1 Score (Weighted): 0.4643
Recall: 0.4858
Log Loss: 1.0721
Brier Score: 0.6365

Classification Report:

              precision    recall  f1-score   support

        Loss       0.50      0.58      0.54       337
        Draw       0.30      0.16      0.21       242
         Win       0.53      0.63      0.57       339

    accuracy                           0.49       918
   macro avg       0.44      0.46      0.44       918
weighted avg       0.46      0.49      0.46       918


Top 10 Most Important Features:
         feature  importance
         is_home    0.074363
   venue_encoded    0.073255
opponent_encoded    0.043710
      attendance    0.037839
      avg_poss_5    0.037468
  avg_xgoal_diff    0.035523
        points_5    0.033793
    team_encoded    0.033556
      avg_pk_acc    0.033506
   avg_goal_diff    0.033430


# Model Comparison

In [43]:
# Compute baseline (majority class)

majority_class=y_train.value_counts().idxmax()
baseline_pred=np.full(len(y_test), majority_class)
baseline_acc=accuracy_score(y_test, baseline_pred)
outcome_map={-1: 'Loss', 0: 'Draw', 1: 'Win'}
majority_name=outcome_map[majority_class]

print(f"Baseline Performance always predicting '{majority_name}' class:")
print(f"Accuracy: {baseline_acc:.4f} ({baseline_acc*100:.2f}%)")

results_df=pd.DataFrame(results)
results_df=results_df.sort_values('F1 Score (Macro)', ascending=False)

print("\nModel performance summary: \n")
print(results_df.to_string(index=False))

# Find best model based on F1 Score (Macro)
best_model_name=results_df.iloc[0]['Model']
best_accuracy=results_df.iloc[0]['Accuracy']
best_f1_macro=results_df.iloc[0]['F1 Score (Macro)']

print(f"\nBest model: {best_model_name} with Accuracy: {best_accuracy:.4f} and F1 Score (Macro): {best_f1_macro:.4f}")

Baseline Performance always predicting 'Win' class:
Accuracy: 0.3693 (36.93%)

Model performance summary: 

              Model  Accuracy  Precision  F1 Score (Macro)  F1 Score (Weighted)   Recall  Log Loss  Brier Score
            XGBoost  0.485839   0.459517          0.440263             0.464296 0.485839  1.072110     0.636513
Logistic Regression  0.449891   0.437344          0.423060             0.442165 0.449891  1.046677     0.629249
      Random Forest  0.466231   0.438810          0.422720             0.447602 0.466231  1.020427     0.611880

Best model: XGBoost with Accuracy: 0.4858 and F1 Score (Macro): 0.4403


# Save the best Model

In [46]:
output_path=os.path.join('..', 'Models')
if not os.path.exists(output_path):
    os.makedirs(output_path)
# Save models

best_model=models[best_model_name]
model_filename=os.path.join(output_path, f"{best_model_name.replace(' ', '_').lower()}_best_model.pkl")

# Create a dictionary with model and metadata
model_data={
    'model': best_model,
    'Model Name': best_model_name,
    'features': feature_cols,
    'scaler': scaler if best_model_name == 'Logistic Regression' else None,
    'training_date_range': (train_dates.min(), train_dates.max()),
    'testing_date_range': (test_dates.min(), test_dates.max()),
    'accuracy': best_accuracy,
    'f1_macro': best_f1_macro,
    'result' : results_df.to_dict(orient='records')
}
with open(model_filename, 'wb') as f:
    pickle.dump(model_data, f) 
print(f"Best model saved to: {model_filename}")
print(f"Model Type: {best_model_name}")

#Also save all models
for model_name, model in models.items():
    model_file=os.path.join(output_path, f"{model_name.replace(' ', '_').lower()}_model.pkl")
    model_data={
        'model': model,
        'features': feature_cols,
        'scaler': scaler,
        'training_date_range': (train_dates.min(), train_dates.max()),
        'testing_date_range': (test_dates.min(), test_dates.max()),
        'result' : results_df.to_dict(orient='records'),
        'confusion_matrices' : confusion_matrices
    }
    with open(model_file, 'wb') as f:
        pickle.dump(model_data, f)
    print(f" All models saved to: {model_file}")

# Save feature importance for tree models
if best_model_name in ['Random Forest', 'XGBoost']:
    importance_df= feature_importance_rf if best_model_name=='Random Forest' else feature_importance_xgb
    importance_file=os.path.join(output_path,"feature_importance.csv")
    importance_df.to_csv(importance_file, index=False)
    print(f"Feature importance saved to: {importance_file}")

Best model saved to: ..\Models\xgboost_best_model.pkl
Model Type: XGBoost
 All models saved to: ..\Models\logistic_regression_model.pkl
 All models saved to: ..\Models\random_forest_model.pkl
 All models saved to: ..\Models\xgboost_model.pkl
Feature importance saved to: ..\Models\feature_importance.csv
